# L09. Object Detection with YOLO

pretrained YOLO 모델로 이미지에서 객체를 검출하고 bounding box, confidence, class를 직접 읽어봅니다.


## 1. Detection이란?

object detection은 이미지 안에 있는 객체의 위치와 종류를 함께 찾는 문제입니다. 출력에는 보통 class, bounding box, confidence가 포함됩니다.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'


## 2. 준비

이 실습은 `ultralytics` 패키지를 사용합니다. 필요한 경우 설치 셀의 주석을 해제하고 한 번만 실행하세요.


In [ ]:
# 필요한 경우 한 번만 실행하세요.
# %pip install ultralytics requests pillow matplotlib


In [ ]:
import requests
from ultralytics import YOLO

sample_dir = Path('data/yolo_samples')
sample_dir.mkdir(parents=True, exist_ok=True)

image_urls = {
    'bus.jpg': 'https://ultralytics.com/images/bus.jpg',
    'zidane.jpg': 'https://ultralytics.com/images/zidane.jpg',
}

for filename, url in image_urls.items():
    path = sample_dir / filename
    if not path.exists():
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        path.write_bytes(response.content)
        print('downloaded', path)
    else:
        print('exists', path)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, filename in zip(axes, image_urls.keys()):
    image = Image.open(sample_dir / filename)
    ax.imshow(image)
    ax.set_title(filename)
    ax.axis('off')

plt.tight_layout()
plt.show()


## 3. Pretrained YOLO 추론

가벼운 pretrained 모델 `yolov8n.pt`를 사용합니다.


In [ ]:
model = YOLO('yolov8n.pt')
results = model(
    [str(sample_dir / 'bus.jpg'), str(sample_dir / 'zidane.jpg')],
    verbose=False,
)

print('number of result objects:', len(results))


In [ ]:
for result in results:
    rendered = result.plot()
    plt.figure(figsize=(6, 4))
    plt.imshow(rendered[..., ::-1])
    plt.title(Path(result.path).name)
    plt.axis('off')
    plt.show()


## 4. Detection 결과 읽기

각 box에 대해 좌표, confidence, class를 확인합니다.


In [ ]:
for result in results:
    print('\nimage:', Path(result.path).name)
    boxes = result.boxes

    for i in range(len(boxes)):
        xyxy = boxes.xyxy[i].tolist()
        conf = float(boxes.conf[i])
        cls_id = int(boxes.cls[i])
        cls_name = result.names[cls_id]
        rounded_box = list(map(lambda v: round(v, 1), xyxy))
        print(f' box={rounded_box}, conf={conf:.3f}, class={cls_name}')


In [ ]:
result = results[0]
image = Image.open(result.path)

fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(image)

for i in range(len(result.boxes)):
    x1, y1, x2, y2 = result.boxes.xyxy[i].tolist()
    conf = float(result.boxes.conf[i])
    cls_id = int(result.boxes.cls[i])
    cls_name = result.names[cls_id]

    rect = patches.Rectangle(
        (x1, y1),
        x2 - x1,
        y2 - y1,
        linewidth=2,
        edgecolor='lime',
        facecolor='none',
    )
    ax.add_patch(rect)
    ax.text(
        x1,
        max(y1 - 5, 5),
        f'{cls_name} {conf:.2f}',
        color='yellow',
        fontsize=9,
        bbox=dict(facecolor='black', alpha=0.5, pad=2),
    )

ax.set_title('Detection Result with Bounding Boxes')
ax.axis('off')
plt.tight_layout()
plt.show()


## 5. Fine-tuning 예시

실제 프로젝트에서는 pretrained YOLO를 바로 쓰거나, 작은 데이터셋으로 추가 학습합니다. 아래는 아주 짧은 fine-tuning 예시입니다.


In [ ]:
# 실행 시간이 걸릴 수 있으므로 필요할 때만 실행하세요.
#
# train_model = YOLO('yolov8n.pt')
# train_model.train(data='coco8.yaml', epochs=1, imgsz=640, batch=8)


## 6. 핵심 정리

- YOLO는 box, class, confidence를 함께 예측합니다.
- pretrained 모델만으로도 detection을 바로 실험할 수 있습니다.
- detection 결과를 직접 읽어보면 box 좌표와 confidence의 의미가 분명해집니다.
- 필요하면 작은 공개 데이터셋으로 fine-tuning할 수 있습니다.

## 7. 연습 문제

1. 두 이미지에서 어떤 객체가 검출되었는지 직접 정리해보세요.
2. confidence threshold를 바꾸면 검출 결과가 어떻게 달라지는지 확인해보세요.

```python
results = model(str(sample_dir / 'bus.jpg'), conf=0.5)
```

3. 다른 공개 이미지를 하나 더 다운로드해 YOLO를 실행해보세요.
4. `yolov8n.pt` 대신 더 큰 모델을 써보세요.

```python
model = YOLO('yolov8s.pt')
```

5. fine-tuning 셀을 실제로 실행할 경우 어떤 파일과 로그가 생기는지 확인해보세요.


## 8. 연습문제 풀이

아래 셀들은 앞에서 만든 `model`, `results`, `sample_dir`, `image_urls`를 그대로 사용합니다.


### Exercise 1 풀이

검출된 객체를 이미지별로 정리합니다. 실제 항목은 모델 버전과 confidence threshold에 따라 조금 달라질 수 있으므로, 아래 코드로 현재 실행 결과를 표처럼 확인합니다.


In [ ]:
def summarize_detections(results):
    rows = []
    for result in results:
        boxes = result.boxes
        if boxes is None:
            continue
        for i in range(len(boxes)):
            cls_id = int(boxes.cls[i])
            rows.append({
                'image': Path(result.path).name,
                'class': result.names[cls_id],
                'confidence': float(boxes.conf[i]),
                'box': [round(v, 1) for v in boxes.xyxy[i].tolist()],
            })
    return rows

detection_rows = summarize_detections(results)
for row in detection_rows:
    print(f"{row['image']:10s} | {row['class']:12s} | conf={row['confidence']:.3f} | box={row['box']}")


### Exercise 2 풀이

confidence threshold를 높이면 낮은 확신도의 box가 제거되어 검출 개수는 줄어드는 경향이 있습니다.


In [ ]:
def count_by_class(result):
    counts = {}
    boxes = result.boxes
    if boxes is None:
        return counts
    for i in range(len(boxes)):
        cls_id = int(boxes.cls[i])
        cls_name = result.names[cls_id]
        counts[cls_name] = counts.get(cls_name, 0) + 1
    return counts

for conf in [0.25, 0.50, 0.70]:
    threshold_result = model(str(sample_dir / 'bus.jpg'), conf=conf, verbose=False)[0]
    total = 0 if threshold_result.boxes is None else len(threshold_result.boxes)
    print(f'conf={conf:.2f} -> boxes={total}, classes={count_by_class(threshold_result)}')


### Exercise 3 풀이

다른 공개 이미지로 YOLO를 실행합니다. 아래 예시는 Darknet 예제 이미지인 `dog.jpg`를 사용합니다.


In [ ]:
extra_url = 'https://raw.githubusercontent.com/pjreddie/darknet/master/data/dog.jpg'
extra_path = sample_dir / 'dog.jpg'

if not extra_path.exists():
    response = requests.get(extra_url, timeout=30)
    response.raise_for_status()
    extra_path.write_bytes(response.content)
    print('downloaded', extra_path)
else:
    print('exists', extra_path)

extra_results = model(str(extra_path), verbose=False)
extra_rendered = extra_results[0].plot()

plt.figure(figsize=(6, 4))
plt.imshow(extra_rendered[..., ::-1])
plt.title('dog.jpg detection')
plt.axis('off')
plt.show()

for row in summarize_detections(extra_results):
    print(f"{row['class']:12s} | conf={row['confidence']:.3f} | box={row['box']}")


### Exercise 4 풀이

`yolov8s.pt`는 `yolov8n.pt`보다 큰 모델입니다. 보통 더 무겁지만 더 안정적인 검출을 기대할 수 있습니다.


In [ ]:
small_result = model(str(sample_dir / 'bus.jpg'), verbose=False)[0]
large_model = YOLO('yolov8s.pt')
large_result = large_model(str(sample_dir / 'bus.jpg'), verbose=False)[0]

print('yolov8n boxes:', 0 if small_result.boxes is None else len(small_result.boxes), count_by_class(small_result))
print('yolov8s boxes:', 0 if large_result.boxes is None else len(large_result.boxes), count_by_class(large_result))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(small_result.plot()[..., ::-1])
axes[0].set_title('yolov8n.pt')
axes[0].axis('off')
axes[1].imshow(large_result.plot()[..., ::-1])
axes[1].set_title('yolov8s.pt')
axes[1].axis('off')
plt.tight_layout()
plt.show()


### Exercise 5 풀이

fine-tuning을 실행하면 기본적으로 `runs/detect/train...` 아래에 학습 결과가 저장됩니다. 주요 파일은 다음과 같습니다.

- `weights/best.pt`: validation 기준 가장 좋은 모델
- `weights/last.pt`: 마지막 epoch 모델
- `results.csv`: epoch별 loss와 metric 기록
- `args.yaml`: 학습 설정
- `train_batch*.jpg`, `val_batch*.jpg`: 학습/검증 배치 시각화


In [ ]:
runs_dir = Path('runs/detect')
if runs_dir.exists():
    for path in sorted(runs_dir.glob('train*')):
        print(path)
        for child in sorted(path.glob('*'))[:10]:
            print('  ', child.name)
else:
    print('아직 fine-tuning을 실행하지 않아 runs/detect 폴더가 없습니다.')
